# 02 Exploratory Analysis


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    markers = ("src", "data", "notebooks")

    for p in [start, *start.parents]:
        if p.name == "customer-segmentation-analytics" and all((p / m).is_dir() for m in markers):
            return p

    for p in [start, *start.parents]:
        candidate = p / "Improvements" / "Statistics" / "customer-segmentation-analytics"
        if all((candidate / m).is_dir() for m in markers):
            return candidate

    raise RuntimeError("Could not locate customer-segmentation-analytics project root.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")


## Objective


Understand distributions, customer segment mix, and behaviour-value patterns.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "customers_clean.parquet")
print("Dataset shape:", df.shape)
df.head()


## Sales Distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["avg_monthly_spend"], bins=40, kde=True)
plt.title("Distribution of Average Monthly Spend")
plt.xlabel("avg_monthly_spend")
plt.tight_layout()
plt.show()


## Orders per Customer (Purchase Frequency)


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["purchase_frequency"], bins=40, kde=True)
plt.title("Distribution of Purchase Frequency")
plt.xlabel("purchase_frequency")
plt.tight_layout()
plt.show()


## Revenue by Region


In [ ]:
region_spend = df.groupby("region", dropna=False)["avg_monthly_spend"].mean().sort_values(ascending=False)
region_spend


In [ ]:
plt.figure(figsize=(10, 5))
region_spend.plot(kind="bar")
plt.title("Average Monthly Spend by Region")
plt.ylabel("avg_monthly_spend")
plt.tight_layout()
plt.show()


## Segment Composition


In [ ]:
segment_share = df["customer_segment"].value_counts(normalize=True).mul(100).round(2)
segment_share


## Correlation View (Numeric Features)


In [ ]:
numeric_cols = [
    "age",
    "annual_income",
    "months_active",
    "avg_monthly_spend",
    "purchase_frequency",
    "avg_order_value",
    "discount_usage_rate",
    "return_rate",
    "browsing_time_minutes",
    "support_interactions",
]

corr = df[numeric_cols].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()
